In [34]:
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import equivariant steerable cnn
import escnn.gspaces
import escnn.nn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import from_numpy

from tqdm import tqdm
# Checking if you have a GPU and CUDA set up -- if not, training will take longer by running on the CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
# Set the plotting style
#plt.style.use('seaborn-poster')

from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from os import listdir
from os.path import isfile, join

from copy import deepcopy

from getdist import MCSamples, plots

import pickle as pk

from hyperparameters import hparams

In [35]:
ens = 20 # Turn this up
path = '/project/ag-weller/Kai.Lehman/gaussian_fields/'
onlyfiles = [f[:-4] for f in listdir(path) if f[-3:] == 'npy']
onlyfiles_split = np.array([s.split('_') for s in onlyfiles])
stds = onlyfiles_split[:,8].astype(np.float32)
corrlens = onlyfiles_split[:,10].astype(np.float32)

imgs = np.zeros((ens*len(onlyfiles),100,100))
for i, name in enumerate(onlyfiles):
    img_stack = np.load(path+name+'.npy')
    imgs[i*ens:(i+1)*ens] = img_stack


imgs = from_numpy(np.reshape(imgs.astype(np.float32),((ens*len(onlyfiles),1,100,100)))).to('cuda')

In [36]:
stds_tiled = (np.ones((ens,len(stds)),dtype=np.float32)*stds).T.flatten()
corrlens_tiled = (np.ones((ens,len(corrlens)),dtype=np.float32)*corrlens).T.flatten()

# building one theta vector
theta = from_numpy(np.vstack((stds_tiled,corrlens_tiled)).T).to('cuda')
print(theta.shape)

torch.Size([20000, 2])


In [37]:
# set up C3NN
bandlimit = 6
gspace = escnn.gspaces.rot2dOnR2(N=-1, maximum_frequency=bandlimit)


""" Implements the non-linear convolution operation using the recursive formula."""
class NonlinearConv(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order):
        super().__init__()
        self.order = order
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size

        """
        # Initialize filter similar as to how Pytorch initializes nn.Conv2d, but all positive
        # This is the old way of initializing C3NN filters without rotational equivariance/invariance
        weight_sample_k = 1.0 / np.sqrt(filter_size**2 * in_channels)
        init_filt = (2 * weight_sample_k * torch.rand(num_filts, in_channels, filter_size, filter_size, dtype=torch.float32))
        self.conv_filt = torch.nn.Parameter(init_filt)
        # Initialize some slices of the filter to be 0
        with torch.no_grad():
            self.conv_filt[:, :, int(self.filter_size/2)-2:int(self.filter_size/2)+3, int(self.filter_size/2)-2:int(self.filter_size/2)+3]=0.0
        """
        
        # Initialize the escnn filter
        feat_type_in = escnn.nn.FieldType(gspace, self.in_channels*[gspace.trivial_repr])
        feat_type_out = escnn.nn.FieldType(gspace, self.num_filts*[gspace.trivial_repr])
        # THIS LAYER KILLS DEEPCOPY
        self.conv = escnn.nn.R2Conv(in_type=feat_type_in, out_type=feat_type_out, bias=False, kernel_size=self.filter_size) 
        
        # to enable deepcopy
        self.detached = False
        
    def forward(self, x):
        # We will work with strictly positive filters for easier interpretation!
        # foll
        if self.detached == False:
            self.abs_filt = self.conv.expand_parameters()[0].abs()
        elif self.detached == True:
            pass
        
        # Convolutions of powers of the input with powers of the absolute valued filter
        conv_powers = [F.conv2d(x ** k, self.abs_filt ** k, stride=1, padding=int((self.filter_size-1)/2.0)) for k in range(1, self.order + 1)]
        
        #troubleshooting:
        #conv_powers = [self.convnormal(x ** k) for k in range(1, self.order + 1)]
        
        # C_0 is defined to be one everywhere
        corrs = [torch.ones_like(conv_powers[0])]

        # Compute each order of C_n recursively
        for n in range(1, self.order + 1):
            Cn = torch.zeros_like(conv_powers[0])
            for k in range(n):
                Cn += (-1) ** k * conv_powers[k] * corrs[n - k - 1]
            Cn /= n
            corrs.append(Cn)

        # Stack correlator maps along the channel dimension, dropping the zeroth order (C_0)
        # Here the channel dimension refers to the number of channels
        #return torch.cat(corrs[1:], dim=-3)
        return torch.cat(corrs[1:], dim=-3)

    def _detach(self):
        del self.conv
        self.abs_filt = self.abs_filt.detach()
        self.detached = True


""" Does a simple average over the last two dimensions of the input."""
class Mean2d(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return x.mean((-2, -1))


""" The full CCNN is Input -> NonlinearConv -> Spatial Avg -> BatchNorm -> Logistic. The way it is constructed
    here assumes a binary classification task, but this can be extended to the multiclass case."""
class CCNN(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order, absbeta=False, slice=False, pool=False):
        super().__init__()
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size
        self.order = order
        self.absbeta = absbeta

        # the model architecture with escnn
        self.corr_conv = NonlinearConv(self.in_channels, self.num_filts, self.filter_size, self.order)
        self.spatial_mean = Mean2d()
        # I preserve the batch normalization layer here so that we can scale the compressed summary statistics to the same order of magnitude
        # This may help the numerical stability of inference in the 2nd part
        self.bn = nn.BatchNorm1d(num_filts * order, affine=True, eps=3.185289e-35, dtype=torch.float32)


        #troubleshooting
        #self.conv = Conv2d(1,1,11)
    
    def forward(self, x, chop=True):    
        
        x = self.corr_conv(x)
        #x = self.conv(x)
        x = self.spatial_mean(x)
        if chop == False:
            x = self.bn(x)


        return x

    def detach(self):
        self.corr_conv._detach()


#=== 2) Set up some things for training ===#
# Number of times the model will see the full dataset
#epochs = 5
# The initial learning rate -- an important hyperparameter
#lr = 0.505631 
#lr = 0.5
# The strength of the regularization loss -- we can make this very strong in these easy toy examples
#gamma = 0.030383
# The number of input channels 
in_channels = 1
# The number of filters to learn in our model
num_filts = 1
# The spatial length of the filters
filter_size = 11
# The highest order of correlation measured
order = 2
# Make our model -- for simplicity, let's train a two-filter model

In [38]:
print(stds.min(),stds.max())
print(corrlens.min(),corrlens.max())

1.0 51.0
5.0 23.0


In [40]:
ccnn = CCNN(in_channels=in_channels, num_filts=num_filts, filter_size=filter_size, order=order, absbeta=False).to(device='cuda')

# Set up prior for sbi
lower = np.array([1.,5.])
upper = np.array([51.,23.])
#lower = np.array([.8,3.])
#upper = np.array([4.,30.])
prior = utils.BoxUniform(low=from_numpy(lower), high=from_numpy(upper),device='cuda')


#set up sbi architecture
density_estimator_custom = posterior_nn(model="maf", hidden_features=hparams.flow_hidden_features, num_transforms=hparams.flow_num_transforms, device='cuda', embedding_net=ccnn)
inference = SNPE_C(prior, device='cuda', density_estimator=density_estimator_custom)

# joint training
density_estimator = inference.append_simulations(theta[:10], imgs[:10]).train(learning_rate=hparams.learning_rate,training_batch_size=hparams.flow_training_batch_size,max_num_epochs=5)
# DANGER DANGER
ccnn.detach()
# now deepcopy works:
test = deepcopy(density_estimator._embedding_net)

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(
/home/k/Kai.Lehman/C3NN_inference/sbi/utils/user_input_checks.py:435: UserWarning: Mismatch between the device of the data fed to the embedding_net and the device of the embedding_net's weights. Fed data has device 'cpu' vs embedding_net weights have device 'cuda:0'. Automatically switching the embedding_net's device to 'cpu', which could otherwise be done manually using the line `embedding_net.to('cpu')`.
  warnings.warn(


 Training neural network. Epochs trained: 6DANGER!!!! No more deepycopy after training!!!


In [17]:
with open(f"inference.pkl", "wb") as handle:
            pk.dump(inference, handle)
with open(f"density_estimator.pkl", "wb") as handle:
            pk.dump(deepcopy(density_estimator), handle)
print('Model saved')

Model saved


/home/k/Kai.Lehman/C3NN_inference/sbi/inference/base.py:429: UserWarning: When the inference object is pickled, the behaviour of the loaded object changes in the following two ways: 1) `.train(..., retrain_from_scratch=True)` is not supported. 2) When the loaded object calls the `.train()` method, it generates a new tensorboard summary writer (instead of appending to the current one).
  warn(


In [5]:
with open('density_estimator.pkl', 'rb') as f:
    density_estimator = pk.load(f)
with open('inference.pkl', 'rb') as f:
    inference = pk.load(f)

In [6]:
train_loader, val_loader = inference.get_dataloaders(
            #start_idx,
            training_batch_size=1,
            #validation_fraction,
            resume_training=False,
            #dataloader_kwargs=dataloader_kwargs,
            dataloader_kwargs={"drop_last": False},
)

theta_test, img_test, _ = next(iter(val_loader))

In [8]:
# build posterior and sample
index = 2000
posterior = inference.build_posterior(deepcopy(density_estimator))
posterior_samples = posterior.sample((100000,), x=img_test).cpu()

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

In [21]:
names = np.array(['\sigma','corrlen'])
params = theta_test[0].cpu()


samples = MCSamples(samples=posterior_samples.numpy(), names=names, labels=names, settings={'ignore_rows':.2},label='sbi, Likelihood-Free')
g = plots.get_subplot_plotter()
#g.settings.lab_fontsize = 17
#g.settings.axes_fontsize = 14.3
#g.settings.legend_fontsize = 14.5
g.triangle_plot([samples], names, title_limit=1, filled=True,
                     markers={'\sigma':params[0],'corrlen':params[1]},
                     marker_args={'c':'black'})

plt.show()
#plt.savefig('test0.png')

Removed 0.2 as burn in


In [26]:
# let's do a more statistical test over the entire validation set

no_val_sims = 100
posterior_samples_all = np.zeros((no_val_sims,100000,2))
params = np.zeros((no_val_sims,2))
for i, (param, observation, _) in enumerate(val_loader):
    params[i] = param.cpu()
    observation.to('cuda')
    posterior_samples_all[i] = posterior.sample((100000,), x=observation).cpu()
    if i%10 == 0:
        print(i)
    if i == no_val_sims-1:
        break

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

0


Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

... (100 outputs left out)

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

In [27]:
print(np.shape(posterior_samples_all))
print(np.shape(params))

(100, 100000, 2)
(100, 2)


In [33]:
plt.clf()
fig, ax = plt.subplots(2)
parts0 = ax[0].violinplot(posterior_samples_all[:no_val_sims,:,0].T-params[:no_val_sims,0],params[:no_val_sims,0],showextrema=False,quantiles=np.tile([.16,.84],(no_val_sims,1)).T)
parts1 = ax[1].violinplot(posterior_samples_all[:no_val_sims,:,1].T-params[:no_val_sims,1],params[:no_val_sims,1],showextrema=False,quantiles=np.tile([.16,.84],(no_val_sims,1)).T)

for i, axis in enumerate(ax):
    axis.axhline(0,color='black')
    axis.set_xlabel(fr'True {names[i]}')
    axis.set_ylabel(fr'Predicted - True {names[i]}')

plt.tight_layout()
plt.savefig('violin.png')